# Remote Sensing Practical using Google Earth Engine

Run this notebook in a new environment by running the following in the terminal:

```
conda create --name remote-sensing python=3.11 pip
conda activate remote-sensing

pip install jupyter matplotlib earthengine-api folium

jupyter lab
```

### Import relevant libraries

In [ ]:
import matplotlib.pyplot as plt
import folium
import numpy as np

### Authenticate Google Earth Engine

- First, create a GEE account with Gmail if you don't have one already: https://earthengine.google.com/
- You will need to register a Google Cloud project for Earth Engine (for academic/non-commercial use) -- link will be prompted below.
- Find your **project ID** in the [Earth Engine Code Editor](https://code.earthengine.google.com/) (top right) or the [Google Cloud console](https://console.cloud.google.com/), and paste it into the cell below.

In [ ]:
import ee

# Trigger the authentication flow.
ee.Authenticate()

# Initialize the library.
ee.Initialize(project='ee-ilyankou') # replace with your own project ID

# Part 1 - Download and Display a Multispectral Image 🛰️

### Define area of interest
* Use Google Maps to get the point coordinate of somewhere in central London
* Note that Google Maps default is (lat,lon) but for GEE we need to use (lon,lat)

In [ ]:
london_coords = (51.51353, -0.11206) # latitude, longitude

# Create a point
london_point = ee.Geometry.Point(london_coords[::-1]) # longitude, latitude

# Create a 10km buffer around the point, and get a bounding box
london = london_point.buffer(10000).bounds()

### Download a multispectral image for the area
* In Google Earth Engine, data is organised in collections; each collection contains many images (from different places and different time periods). We are usually interested in one (or a few) images in the collection only
* Use the Search bar in Google Earth Engine to find your preferred provider (Landsat, Sentinel...). We'll use Landsat 9, Collection 2, Tier 1, TOA Reflectance which is `LANDSAT/LC09/C02/T1_TOA`
* TOA stands for *top-of-atmosphere* reflectance: the light measured by the satellite, calibrated to reflectance, but **not** corrected for the effects of the atmosphere (haze, aerosols, water vapour). Atmospherically corrected data is called *surface reflectance* (for Landsat 9, `LANDSAT/LC09/C02/T1_L2`)
* We can narrow down a collection using filters, such as `filterBounds()`, `filterDate()`, filter by cloud coverage, etc
* Once we define the criteria, we can sort the matching images by cloud cover and take the least cloudy one with `.first()`
* Note that `CLOUD_COVER` describes the whole Landsat scene (~185 x 180 km), not just our area of interest -- a scene with low cloud cover can still have clouds over London. For more careful work, clouds can be masked pixel by pixel using the `QA_PIXEL` band
* If no image matches your filters, `.first()` returns nothing and you will get a confusing error later on -- try relaxing the date range or cloud cover threshold

In [ ]:
# First, define your collection
l9_collection = ee.ImageCollection('LANDSAT/LC09/C02/T1_TOA')

# Then, load a single representative image from the collection
l9_img = ee.Image(
    l9_collection
    .filterBounds(london) # get only those images that contain our boundary for London
    .filterDate('2026-01-01', '2026-11-05') # get images from 2026
    .filter( ee.Filter.lt('CLOUD_COVER', 10) )
    .sort('CLOUD_COVER') # sort in ascending order by cloud cover
    .first() # get the first image only (least cloudy)
    .clip(london) # clip to the London area extent
)

### List all available bands
Use Google Earth Engine GUI (Description and Band tabs) to learn what each band is

For Landsat 9,
* B2 = Blue
* B3 = Green
* B4 = Red
* B5 = Near infrared (NIR)
* B6 and B7 are shortwave infrared (SWIR)

In [ ]:
l9_img.bandNames().getInfo()

### Create a true-color composite image and show on map
* We can combine red, green, and blue band values to create a true-colour composite image
* Use `folium` to place the raster image on an interactive map

In [ ]:
# Define a method for displaying Earth Engine image tiles to folium map
# Adapted from https://developers.google.com/earth-engine/tutorials/community/detecting-changes-in-sentinel-1-imagery-pt-1
def add_ee_layer(self, ee_image_object, vis_params):
    map_id_dict = ee.Image(ee_image_object).getMapId(vis_params)
    folium.raster_layers.TileLayer(
        tiles = map_id_dict['tile_fetcher'].url_format,
        attr = 'Map Data &copy; <a href="https://earthengine.google.com/">Google Earth Engine</a>',
        overlay = True
    ).add_to(self)

# Add EE drawing method to folium
folium.Map.add_ee_layer = add_ee_layer

In [ ]:
red = l9_img.select('B4')
green = l9_img.select('B3')
blue = l9_img.select('B2')

rgb_img = ee.Image.rgb(red, green, blue)

In [ ]:
# Create the map object
m1 = folium.Map(location=london_coords, zoom_start=11)

# Add the layer
m1.add_ee_layer(rgb_img, {
    'min': 0,
    'max': 0.25
})

# Display the map
m1

### Exercise 1 🗺️

Create a true-colour composite layer for a place of your choice (not London) using a Sentinel-2 collection. Display it in a Folium map

Hints for Sentinel-2:
* Use `COPERNICUS/S2_SR_HARMONIZED` (surface reflectance) or `COPERNICUS/S2_HARMONIZED` (top-of-atmosphere). The older `COPERNICUS/S2` collection is deprecated
* The cloud cover property is called `CLOUDY_PIXEL_PERCENTAGE` (not `CLOUD_COVER`)
* Band values are scaled by 10,000, so use something like `'max': 3000` instead of `0.25` when visualising
* Bands: B2 = Blue, B3 = Green, B4 = Red, **B8 = NIR** (not B5, as in Landsat 9) -- you will need this for Exercises 2 and 3

# Part 2. Calculating Normalised Difference Vegetation Index 🌳

Remember the formula for NDVI is: 

`NDVI = (NIR - Red) / (NIR + Red)`

And in Landsat 9 imagery, NIR is band B5, and Red is Band B4

In [ ]:
nir = l9_img.select('B5')

In [ ]:
ndvi_img = nir.subtract(red).divide( nir.add(red) )

In [ ]:
m2 = folium.Map(location=london_coords, zoom_start=11)

m2.add_ee_layer(ndvi_img, {
    'min': -1,
    'max': 1,
    'palette': ['blue', 'white', 'green']
})

# Display the map
m2

#### Let's now create a mask to only show pixels with high NDVI
* This should give us a raster layer of parks and other green areas
* Try changing the default (0.45) value

In [ ]:
ndvi_img_mask = ndvi_img.updateMask(ndvi_img.gt(0.45))

m3 = folium.Map(location=london_coords, zoom_start=11)

m3.add_ee_layer(ndvi_img_mask, {
    'min': -1,
    'max': 1,
    'palette': ['green']
})

m3

### Exercise 2 💧

Create a *Normalised Difference Water Index* for your study area from Exercise 1, and plot it in a folium map

You can use the formula `NDWI = (Green − NIR) / (Green + NIR)`, where positive values (NDWI > 0) usually correspond to water bodies. Remember to use the correct NIR band for your sensor!

# Part 3 - Change Detection using Simple Difference 🕵️‍♀️

Let's take two images, the least cloudy one from summer 2025 and the least cloudy one from summer 2026, and detect changes in NDVI using simple difference between the index values.

In [ ]:
l9_img_2025 = ee.Image(
    l9_collection
    .filterBounds(london) # get only those images that contain our boundary for London
    .filterDate('2025-06-01', '2025-08-31') # get images from summer 2025
    .sort('CLOUD_COVER') # sort in ascending order by cloud cover
    .first() # get the first image only (least cloudy)
    .clip(london) # clip to the London area extent
)

l9_img_2026 = ee.Image(
    l9_collection
    .filterBounds(london) # get only those images that contain our boundary for London
    .filterDate('2026-06-01', '2026-08-31') # get images from summer 2026
    .sort('CLOUD_COVER') # sort in ascending order by cloud cover
    .first() # get the first image only (least cloudy)
    .clip(london) # clip to the London area extent
)

### Let's see which images we got
* Display cloud cover and dates for both images

In [ ]:
def get_img_info(img):
    return (
        'Image from '
        + img.date().format('YYYY-MM-dd').getInfo()
        + ' with the cloud cover of '
        + str(img.get('CLOUD_COVER').getInfo())
        + '%'
    )

In [ ]:
get_img_info(l9_img_2025)

In [ ]:
get_img_info(l9_img_2026)

🤔 **Think about it:** the two images are from different dates within the summer, and may have been captured from different Landsat paths (i.e. at a different viewing angle). How much of the "change" we detect below might be due to seasonal differences in vegetation, sun angle, or atmospheric conditions, rather than real changes on the ground?

In [ ]:
# Calculate NDVI indices using a built-in function
ndvi_2025 = l9_img_2025.normalizedDifference(['B5', 'B4'])
ndvi_2026 = l9_img_2026.normalizedDifference(['B5', 'B4'])

# Subtract the two
ndvi_diff = ndvi_2026.subtract(ndvi_2025)

In [ ]:
# Let's get a sample of the NDVI difference values into a 1D numpy array
values_2d = np.array(
    ndvi_diff.sampleRectangle(
        region=london_point.buffer(2000)
    )
    .getInfo()
    ['properties']
    ['nd']
)

values_1d = values_2d.reshape(-1)

In [ ]:
plt.hist(values_1d, bins=50)

* We can see the difference values are more or less normally distributed around 0
* Let's plot differences of **+0.2 or more** in **green** to signify areas that gained vegetation in 2026
* Let's plot differences of **−0.2 or less** in **red** to represent areas that lost vegetation in 2026
* To focus on vegetation, we only consider a loss where there was vegetation in 2025, and a gain where there is vegetation in 2026 (NDVI > 0.3). This way, newly vegetated areas are not hidden

In [ ]:
m4 = folium.Map(location=london_coords, zoom_start=11)

# Consider NDVI values of 0.3+ as vegetation
veg_2025 = ndvi_2025.gt(0.3)
veg_2026 = ndvi_2026.gt(0.3)

# Vegetation gain: NDVI increased by 0.2+ and the pixel is vegetated in 2026
veg_increase = ndvi_diff.updateMask(ndvi_diff.gt(0.2)).updateMask(veg_2026)

# Vegetation loss: NDVI decreased by 0.2+ and the pixel was vegetated in 2025
veg_decrease = ndvi_diff.updateMask(ndvi_diff.lt(-0.2)).updateMask(veg_2025)

# Let's plot these in different colours
m4.add_ee_layer(veg_increase, {'palette': ['green']})
m4.add_ee_layer(veg_decrease, {'palette': ['red']})

m4

### Exercise 3

Create a simple difference map for the NDWI values for the two time periods for your area of choice